Look at snapshots of stratification over time

In [ ]:
import src.utils
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import seaborn as sns
import sklearn.linear_model
import pathlib
import os
import sklearn.linear_model
import cmocean
import cartopy.crs as ccrs

## RNG
rng = np.random.default_rng()

## color palette
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## paths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
# SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Load data

In [ ]:
USE_WIDE = True

## load spatial data
forced, anom_ = src.utils.load_consolidated()

if USE_WIDE:

    ## load wide data
    forced_05, anom_05 = src.utils.load_consolidated_05()

    for v in list(forced_05):
        forced[v] = forced_05[v]
        anom_[v] = anom_05[v]

## get subset of data for total
VARNAMES = ["T", "sst", "taux", "tauy", "w", "u"]
total = anom_[VARNAMES] + forced[VARNAMES]
total = xr.merge([forced[[f"{v}_comp" for v in VARNAMES]], total])
total = src.utils.get_windowed(total, stride=120)

## Load T,h (total)
Th_total = xr.open_dataset(DATA_FP / "cesm" / "Th.nc")
Th = Th_total - Th_total.mean("member")
Th = src.utils.get_windowed(Th, stride=120)

## remove nonlinear rect.

In [ ]:
## specify years
YEARS = total.year

## should we compute by month?
BY_MONTH = True

if BY_MONTH:
    ## get mean and Th
    total_clim = total.sel(year=YEARS).groupby("time.month").mean(["time", "member"])
    Th_sigma = Th["T_34"].groupby("time.month").std("time")
    total_mean = total.groupby("time.month").mean("time")
    total_mean = total_mean[[v for v in list(total) if "comp" not in v]]
    total_mean_comp = total[[v for v in list(total) if "comp" in v]]
    fn = src.utils.estimate_rect_bymonth

else:
    ## get mean and Th
    total_clim = total.sel(year=YEARS).mean(["time", "member"])
    Th_sigma = Th["T_34"].std("time")
    total_mean = total.mean("time")
    total_mean = total_mean[[v for v in list(total) if "comp" not in v]]
    total_mean_comp = total[[v for v in list(total) if "comp" in v]]
    fn = src.utils.estimate_rect


## estimate effect
coefs = fn(xr.merge([total_mean, Th_sigma]), xvar="T_34", yvars=list(total_mean))

## get corrected background state estimate
coefs0 = coefs.sel(degree=0)

## reconstruct background state (un-corrected)
clim = src.utils.reconstruct_wrapper(total_clim, fn=lambda x: x)

## get background state (corrected)
clim_norect = src.utils.reconstruct_wrapper(
    xr.merge([coefs0, total_mean_comp]),
    fn=lambda x: x,
).sel(year=clim.year)

## compute stratification
clim["dTdz"] = clim["T"].differentiate("z_t")
clim_norect["dTdz"] = clim_norect["T"].differentiate("z_t")


## averge over months for plotting
if BY_MONTH:
    # SEL_MONTH=lambda x : x.sel(month=slice(10,12)).mean("month")
    SEL_MONTH = lambda x: x.mean("month")
    clim = SEL_MONTH(clim)
    clim_norect = SEL_MONTH(clim_norect)

## vertical sections

In [ ]:
def plot_sections(data, levels, cmap, levels_diff, cmap_diff="cmo.balance"):
    """plot vertical sections"""

    ## plot setup
    fig, axs = plt.subplots(3, 3, figsize=(10, 7), layout="constrained")

    for j, (Y0, Y1) in enumerate(zip([1870, 2010, 2050], [2010, 2050, 2080])):
    # for j, (Y0, Y1) in enumerate(zip([1870, 1990, 2050], [1990, 2050, 2080])):

        ## baseline
        for i, y in enumerate([Y0, Y1]):
            cp = axs[j, i].contourf(
                data.longitude,
                data.z_t,
                data.sel(year=y),
                cmap=cmap,
                levels=levels,
                extend="both",
            )
            axs[j, i].set_title(y)

        ## difference
        cp_diff = axs[j, 2].contourf(
            data.longitude,
            data.z_t,
            data.sel(year=Y1) - data.sel(year=Y0),
            cmap=cmap_diff,
            levels=levels_diff,
            extend="both",
        )
        axs[j, 2].set_title("Difference")

    ## set ax limit and plot Niño 3.4 bounds
    for ax in axs.flatten():
        ax.set_xlim([140, 280])
        ax.set_ylim([150, 5])
        ax_kwargs = dict(ls="--", c="w", lw=0.8)
        ax.axvline(210, **ax_kwargs)
        ax.axvline(270, **ax_kwargs)
        ax.axhline(80, **ax_kwargs)

    for ax in axs[:, 0]:
        ax.set_yticks([100, 0])
    for ax in axs[:, 1:].flatten():
        ax.set_yticks([])
    for ax in axs[:-1].flatten():
        ax.set_xticks([])
    for ax in axs[-1]:
        ax.set_xticks([140, 210, 270])

    return fig, axs

### $T$

In [ ]:
plot_kwargs = dict(
    cmap="cmo.thermal",
    levels=np.arange(12, 34, 2),
    levels_diff=src.utils.make_cb_range(2, 0.2),
)

fig, axs = plot_sections(clim["T"], **plot_kwargs)

In [ ]:
plot_kwargs = dict(
    cmap="cmo.thermal",
    levels=np.arange(12, 34, 2),
    levels_diff=src.utils.make_cb_range(2, 0.2),
)

fig, axs = plot_sections(clim_norect["T"], **plot_kwargs)

### $\partial T / \partial z$

In [ ]:
plot_kwargs = dict(
    cmap="cmo.amp_r",
    levels=np.arange(-2e-1, 2e-2, 2e-2),
    cmap_diff="cmo.balance_r",
    levels_diff=src.utils.make_cb_range(4e-2, 4e-3),
)

fig, axs = plot_sections(clim["dTdz"], **plot_kwargs)

In [ ]:
plot_kwargs = dict(
    cmap="cmo.amp_r",
    levels=np.arange(-2e-1, 2e-2, 2e-2),
    cmap_diff="cmo.balance_r",
    levels_diff=src.utils.make_cb_range(4e-2, 4e-3),
)

fig, axs = plot_sections(clim_norect["dTdz"], **plot_kwargs)

### $u$

In [ ]:
plot_kwargs = dict(
    cmap="cmo.balance",
    levels=src.utils.make_cb_range(2e6, 2e5),
    cmap_diff="cmo.balance",
    levels_diff=src.utils.make_cb_range(4e5, 4e4),
)

fig, axs = plot_sections(clim["u"], **plot_kwargs)

In [ ]:
plot_kwargs = dict(
    cmap="cmo.balance",
    levels=src.utils.make_cb_range(2e6, 2e5),
    cmap_diff="cmo.balance",
    levels_diff=src.utils.make_cb_range(4e5, 4e4),
)

fig, axs = plot_sections(clim_norect["u"], **plot_kwargs)

### $w$

In [ ]:
plot_kwargs = dict(
    cmap="cmo.balance",
    levels=src.utils.make_cb_range(3e1, 3e0),
    cmap_diff="cmo.balance",
    levels_diff=src.utils.make_cb_range(5e0, 5e-1),
)

fig, axs = plot_sections(clim["w"], **plot_kwargs)

In [ ]:
plot_kwargs = dict(
    cmap="cmo.balance",
    levels=src.utils.make_cb_range(3e1, 3e0),
    cmap_diff="cmo.balance",
    levels_diff=src.utils.make_cb_range(5e0, 5e-1),
)

fig, axs = plot_sections(clim_norect["w"], **plot_kwargs)

## Spatial structure

In [ ]:
def plot_space_sections(data, levels, cmap, levels_diff, cmap_diff="cmo.balance"):
    """plot vertical sections"""

    ## plot setup
    fig = plt.figure(figsize=(15, 4.375), layout="constrained")
    format_func = lambda ax,: src.utils.plot_setup_pac(
        ax, max_lat=20, lon_range=[130, 285]
    )
    axs = src.utils.subplots_with_proj(fig, nrows=3, ncols=3, format_func=format_func)

    for j, (Y0, Y1) in enumerate(zip([1870, 2010, 2050], [2010, 2050, 2080])):

        ## baseline
        for i, y in enumerate([Y0, Y1]):
            cp = axs[j, i].contourf(
                data.longitude,
                data.latitude,
                data.sel(year=y),
                cmap=cmap,
                levels=levels,
                extend="both",
                transform=ccrs.PlateCarree(),
            )
            # axs[j, i].set_title(y)

        ## difference
        cp_diff = axs[j, 2].contourf(
            data.longitude,
            data.latitude,
            data.sel(year=Y1) - data.sel(year=Y0),
            cmap=cmap_diff,
            levels=levels_diff,
            extend="both",
            transform=ccrs.PlateCarree(),
        )
        # axs[j, 2].set_title("Difference")

    kwargs = dict(lats=[-5, 5], lw=1)
    for ax in axs.flatten():
        src.utils.plot_box(ax, lons=[170, 250], c="k", **kwargs)
        src.utils.plot_box(ax, lons=[240, 280], c="w", ls="--", **kwargs)
        src.utils.plot_box(ax, lons=[140, 190], c="w", ls="--", **kwargs)

    return fig, axs

### SST

In [ ]:
plot_kwargs = dict(
    cmap="cmo.thermal",
    levels=np.arange(20, 34, 1),
    levels_diff=src.utils.make_cb_range(2, 0.2),
)

fig, axs = plot_space_sections(clim["sst"], **plot_kwargs)

In [ ]:
plot_kwargs = dict(
    cmap="cmo.thermal",
    levels=np.arange(20, 34, 1),
    levels_diff=src.utils.make_cb_range(2, 0.2),
)

fig, axs = plot_space_sections(clim_norect["sst"], **plot_kwargs)

### $\tau_x$

In [ ]:
plot_kwargs = dict(
    cmap="cmo.balance",
    levels=src.utils.make_cb_range(1e-1, 1e-2),
    levels_diff=src.utils.make_cb_range(1e-2, 1e-3),
)

fig, axs = plot_space_sections(clim["taux"], **plot_kwargs)

In [ ]:
plot_kwargs = dict(
    cmap="cmo.balance",
    levels=src.utils.make_cb_range(1e-1, 1e-2),
    levels_diff=src.utils.make_cb_range(2e-2, 2e-3),
)

fig, axs = plot_space_sections(clim_norect["taux"], **plot_kwargs)

### $\tau_y$

In [ ]:
plot_kwargs = dict(
    cmap="cmo.balance",
    levels=src.utils.make_cb_range(1e-1, 1e-2),
    levels_diff=src.utils.make_cb_range(1e-2, 1e-3),
)

fig, axs = plot_space_sections(-clim["tauy"], **plot_kwargs)

In [ ]:
plot_kwargs = dict(
    cmap="cmo.balance",
    levels=src.utils.make_cb_range(1e-1, 1e-2),
    levels_diff=src.utils.make_cb_range(1e-2, 1e-3),
)

fig, axs = plot_space_sections(-clim_norect["tauy"], **plot_kwargs)